In [1]:
# Author: Om Mohite

import os
import requests
import pandas as pd
from sqlalchemy import URL, create_engine, text
from dotenv import load_dotenv

load_dotenv()

db_password = os.getenv("DB_PASSWORD")
if not db_password:
    raise ValueError("Set DB_PASSWORD in your .env file.")

server_url = URL.create(
    drivername="mysql+pymysql",
    username=os.getenv("DB_USER", "root"),
    password=db_password,
    host=os.getenv("DB_HOST", "localhost"),
    port=int(os.getenv("DB_PORT", "3306")),
)

server_engine = create_engine(server_url, pool_pre_ping=True)

with server_engine.begin() as conn:
    conn.execute(text("CREATE DATABASE IF NOT EXISTS test_case_11"))

server_engine.dispose()

connection_url = URL.create(
    drivername="mysql+pymysql",
    username=os.getenv("DB_USER", "root"),
    password=db_password,
    host=os.getenv("DB_HOST", "localhost"),
    port=int(os.getenv("DB_PORT", "3306")),
    database="test_case_11",
)

engine = create_engine(connection_url, pool_pre_ping=True)

print("Database test_case_11 is ready.")

Database test_case_11 is ready.


In [2]:
# Author: Om Mohite

api_url = "https://jsonplaceholder.typicode.com/posts"

response = requests.get(api_url, timeout=30)
response.raise_for_status()

posts_data = response.json()
posts_df = pd.json_normalize(posts_data)

print("HTTP status:", response.status_code)
print("Records fetched:", len(posts_df))
print("Columns:", posts_df.columns.tolist())

display(posts_df.head())

HTTP status: 200
Records fetched: 100
Columns: ['userId', 'id', 'title', 'body']


,userId,id,title,body
0,1,1,sunt aut facere repellat provident occaecati e...,quia et suscipit\nsuscipit recusandae consequu...
1,1,2,qui est esse,est rerum tempore vitae\nsequi sint nihil repr...
2,1,3,ea molestias quasi exercitationem repellat qui...,et iusto sed quo iure\nvoluptatem occaecati om...
3,1,4,eum et est occaecati,ullam et saepe reiciendis voluptatem adipisci\...
4,1,5,nesciunt quas odio,repudiandae veniam quaerat sunt sed\nalias aut...


In [3]:
# Author: Om Mohite

clean_posts_df = posts_df.copy()

print("Missing values before cleaning:")
display(clean_posts_df.isna().sum().to_frame("missing_count"))

clean_posts_df["userId"] = clean_posts_df["userId"].fillna(0).astype(int)
clean_posts_df["id"] = clean_posts_df["id"].fillna(0).astype(int)
clean_posts_df["title"] = clean_posts_df["title"].fillna("Unknown")
clean_posts_df["body"] = clean_posts_df["body"].fillna("Unknown")

print("Missing values after cleaning:")
display(clean_posts_df.isna().sum().to_frame("missing_count"))

display(clean_posts_df.head())

Missing values before cleaning:


,missing_count
userId,0
id,0
title,0
body,0


Missing values after cleaning:


,missing_count
userId,0
id,0
title,0
body,0


,userId,id,title,body
0,1,1,sunt aut facere repellat provident occaecati e...,quia et suscipit\nsuscipit recusandae consequu...
1,1,2,qui est esse,est rerum tempore vitae\nsequi sint nihil repr...
2,1,3,ea molestias quasi exercitationem repellat qui...,et iusto sed quo iure\nvoluptatem occaecati om...
3,1,4,eum et est occaecati,ullam et saepe reiciendis voluptatem adipisci\...
4,1,5,nesciunt quas odio,repudiandae veniam quaerat sunt sed\nalias aut...


In [4]:
# Author: Om Mohite

clean_posts_df.to_sql(
    name="web_data",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=100,
    method="multi",
)

with engine.connect() as conn:
    loaded_count = conn.execute(
        text("SELECT COUNT(*) FROM web_data")
    ).scalar_one()

print("Rows loaded into MySQL:", loaded_count)

Rows loaded into MySQL: 100


In [5]:
# Author: Om Mohite

with engine.connect() as conn:
    loaded_df = pd.read_sql(
        text("""
            SELECT userId, id, title, body
            FROM web_data
            ORDER BY id
        """),
        conn,
    )

print("Rows in MySQL:", len(loaded_df))
print("Missing values:", loaded_df.isna().sum().sum())
print("Unique post IDs:", loaded_df["id"].nunique())

display(loaded_df.head())

assert len(loaded_df) == 100
assert loaded_df.isna().sum().sum() == 0
assert loaded_df["id"].nunique() == 100

print("TEST CASE 11 API ETL VALIDATION PASSED")

Rows in MySQL: 100
Missing values: 0
Unique post IDs: 100


,userId,id,title,body
0,1,1,sunt aut facere repellat provident occaecati e...,quia et suscipit\nsuscipit recusandae consequu...
1,1,2,qui est esse,est rerum tempore vitae\nsequi sint nihil repr...
2,1,3,ea molestias quasi exercitationem repellat qui...,et iusto sed quo iure\nvoluptatem occaecati om...
3,1,4,eum et est occaecati,ullam et saepe reiciendis voluptatem adipisci\...
4,1,5,nesciunt quas odio,repudiandae veniam quaerat sunt sed\nalias aut...


TEST CASE 11 API ETL VALIDATION PASSED
